# 06 — Signal Generation

---

## Objective

The objective is to transform Bayesian mispricing estimates into systematic trading signals.

In the previous phase, we estimated:

$ Mispricing = P_{Bayes} - P_{Market} $ 

where:

- $ P_{Bayes} $ is the Bayesian fair probability.
- $ P_{Market} $ is the market-implied probability.

This notebook defines when a mispricing estimate is large enough to generate an actionable signal.

---

## Research Question

Can Bayesian fair-value deviations be transformed into systematic trading signals?

---

## Signal Intuition

If: $ P_{Bayes} > P_{Market} $

then the market may be underpricing the event.

This suggests a potential:

- BUY signal If:
$ P_{Bayes} <P_{Market} $
	​
then the market may be overpricing the event.

This suggests a potential: SELL signal.

If the difference is small, the signal is likely noise.

This suggests:

NO TRADE 

Signal Rule

A simple threshold-based rule will be used:

$ Signal = \begin{cases} BUY, & Mispricing > \tau \\  SELL, & Mispricing < -\tau \\  NO\ TRADE, & otherwise \end{cases}$


where: $ τ $ 

is the minimum mispricing threshold required to enter a trade.

Key Questions


1. How many signals are generated for different thresholds?
2. Are signals concentrated in specific probability regions?
3. Do BUY and SELL signals appear balanced?
4. Are signal magnitudes large enough to justify trading?
5. Which markets generate the strongest signals?


In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df_signal = pd.read_csv("../data/processed/market_dataset_with_fair_value.csv")
    

Missing columns: []


,market_id,question,final_probability,bayesian_fair_probability,mispricing,outcome
0,248594,Will Hunter Biden be federally indicted by May...,0.020,0.038462,0.018462,0
1,249778,MLB: Cincinnati Reds vs. Pittsburgh Pirates 20...,0.550,0.600000,0.050000,1
2,250474,Did US GDP grow 2.5% or more in Q1 2023?,0.360,0.333333,-0.026667,0
3,251025,Will Ron DeSantis file to run for president by...,0.995,0.900000,-0.095000,1
4,251027,USD/TRY (Turkish Lira) above 19.75 on May 22?,0.825,0.666667,-0.158333,1


In [2]:
def generate_signal(mispricing, threshold=0.05):
    if mispricing > threshold:
        return "BUY"
    elif mispricing < -threshold:
        return "SELL"
    else:
        return "NO TRADE"


threshold = 0.05

df_signal["signal"] = df_signal["mispricing"].apply(lambda x: generate_signal(x, threshold=threshold))

print("signal distribution:")
print(df_signal["signal"].value_counts())

display(df_signal[["market_id","question","final_probability","bayesian_fair_probability","mispricing",
            "signal","outcome"]].sort_values("mispricing", ascending=False).head(5))

signal distribution:
signal
NO TRADE    28
SELL        12
BUY          3
Name: count, dtype: int64


,market_id,question,final_probability,bayesian_fair_probability,mispricing,signal,outcome
7,252047,Will Dillon Danis post 5+ pics of Nina on Aug ...,0.2350,0.500000,0.265000,BUY,1
19,253384,Elizabeth Warren crypto bill become law by June?,0.2350,0.500000,0.265000,BUY,0
42,255410,US inflation >0.4% from Feb to March 2024?,0.5350,0.600000,0.065000,BUY,0
1,249778,MLB: Cincinnati Reds vs. Pittsburgh Pirates 20...,0.5500,0.600000,0.050000,NO TRADE,1
30,254573,Will Fed cut interest rates 1 time in 2024?,0.0005,0.038462,0.037962,NO TRADE,0
